# L14: Transformer与注意力机制


# L14: Transformer与注意力机制

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：
1. 解释seq2seq架构的局限性与引入注意力机制的动机
2. 理解自注意力机制（Self-Attention）的计算过程与数学原理
3. 掌握Transformer的整体架构：编码器-解码器、多头注意力、位置编码
4. 能手算注意力机制的矩阵运算过程
5. 理解Transformer相对于RNN的优势：并行化、长距离依赖、可解释性

## 2. 核心知识点

### 2.1 seq2seq与注意力机制

**seq2seq模型**:


In [ ]:
输入序列 → Encoder(RNN) → 固定向量 → Decoder(RNN) → 输出序列


**问题**: 固定维度的上下文向量成为信息瓶颈，特别是处理长序列时。

**注意力机制解决方案**:
- 不再将整个输入压缩为单一向量
- Decoder每步生成时，可"关注"输入序列的不同部分
- 通过注意力权重动态分配权重


In [ ]:
# 注意力机制伪代码
def attention(decoder_hidden, encoder_outputs):
    """
    decoder_hidden: Decoder当前隐藏状态 (hidden_dim,)
    encoder_outputs: 所有Encoder隐藏状态 (seq_len, hidden_dim)
    """
    # 计算注意力分数
    scores = []
    for i in range(seq_len):
        # 计算Decoder与每个Encoder输出的相似度
        score = dot_product(decoder_hidden, encoder_outputs[i])
        scores.append(score)
    
    # softmax归一化得到注意力权重
    attention_weights = softmax(scores)
    
    # 加权求和得到上下文向量
    context = sum(attention_weights[i] * encoder_outputs[i] for i in range(seq_len))
    
    return context, attention_weights


### 2.2 自注意力机制（Self-Attention）

**核心思想**: 序列内部任意位置可以关注序列中所有其他位置

**三个向量**: Query (Q), Key (K), Value (V)
- Q: 我在寻找什么
- K: 我包含什么信息
- V: 当匹配时我提供什么信息

**计算过程**:


In [ ]:
1. 输入X通过三个线性变换得到Q、K、V
   Q = X · Wq
   K = X · Wk
   V = X · Wv

2. 计算注意力分数
   scores = Q · Kᵀ / √d_k

3. softmax归一化
   attention_weights = softmax(scores, dim=-1)

4. 加权求和
   output = attention_weights · V


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(SelfAttention, self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads
        
        assert self.head_dim * heads == embed_size, "embed_size must be divisible by heads"
        
        self.Wq = nn.Linear(embed_size, embed_size)
        self.Wk = nn.Linear(embed_size, embed_size)
        self.Wv = nn.Linear(embed_size, embed_size)
        self.fc_out = nn.Linear(embed_size, embed_size)
        
    def forward(self, x, mask=None):
        batch_size, seq_length, embed_size = x.shape
        
        # 线性变换
        Q = self.Wq(x)  # (batch, seq_len, embed_size)
        K = self.Wk(x)
        V = self.Wv(x)
        
        # 分头
        Q = Q.reshape(batch_size, seq_length, self.heads, self.head_dim).permute(0, 2, 1, 3)
        K = K.reshape(batch_size, seq_length, self.heads, self.head_dim).permute(0, 2, 1, 3)
        V = V.reshape(batch_size, seq_length, self.heads, self.head_dim).permute(0, 2, 1, 3)
        
        # 注意力分数
        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.head_dim)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attention_weights = F.softmax(scores, dim=-1)
        
        # 加权求和
        context = torch.matmul(attention_weights, V)
        context = context.permute(0, 2, 1, 3).reshape(batch_size, seq_length, embed_size)
        
        return self.fc_out(context), attention_weights

# 演示
batch_size = 2
seq_len = 5
embed_size = 8
heads = 2

x = torch.randn(batch_size, seq_len, embed_size)
attention = SelfAttention(embed_size, heads)
output, weights = attention(x)

print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}")
print(f"注意力权重形状: {weights.shape}")


### 2.3 Transformer架构

**编码器 (Encoder)**:
1. Input Embedding + Positional Encoding
2. Multi-Head Self-Attention
3. Add & LayerNorm
4. Feed Forward
5. Add & LayerNorm

**解码器 (Decoder)**:
1. Output Embedding + Positional Encoding
2. Masked Multi-Head Self-Attention（不能看到未来）
3. Encoder-Decoder Attention（关注Encoder输出）
4. Feed Forward
5. Add & LayerNorm

**位置编码 (Positional Encoding)**:


In [ ]:
import numpy as np

def positional_encoding(seq_length, d_model):
    """正弦和余弦位置编码"""
    PE = np.zeros((seq_length, d_model))
    
    for pos in range(seq_length):
        for i in range(0, d_model, 2):
            # 偶数位置：正弦
            PE[pos, i] = np.sin(pos / (10000 ** (i / d_model)))
            # 奇数位置：余弦
            PE[pos, i + 1] = np.cos(pos / (10000 ** ((i + 1) / d_model)))
    
    return PE

# 可视化位置编码
import matplotlib.pyplot as plt

seq_len, d_model = 100, 64
pe = positional_encoding(seq_len, d_model)

plt.figure(figsize=(12, 5))
plt.imshow(pe.T, cmap='RdBu', aspect='auto')
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Positional Encoding Heatmap')
plt.colorbar()
plt.savefig("l14_positional_encoding.png", dpi=150)
plt.show()


### 2.4 多头注意力与Transformer实现


In [ ]:
"""
L14-transformer.py
简化版Transformer实现
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads, dropout, forward_expansion):
        super(TransformerBlock, self).__init__()
        self.attention = SelfAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size)
        )
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        # 自注意力
        attention_out, _ = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attention_out))
        
        # 前馈网络
        forward_out = self.feed_forward(x)
        x = self.norm2(x + self.dropout(forward_out))
        
        return x

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, src_pad_idx, trg_pad_idx,
                 embed_size=256, num_layers=6, heads=8, dropout=0.2, 
                 forward_expansion=4, max_length=100):
        super(Transformer, self).__init__()
        
        self.src_embed = nn.Embedding(src_vocab_size, embed_size)
        self.trg_embed = nn.Embedding(trg_vocab_size, embed_size)
        self.pos_encoder = nn.Embedding(max_length, embed_size)
        
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(embed_size, heads, dropout, forward_expansion)
            for _ in range(num_layers)
        ])
        
        self.fc_out = nn.Linear(embed_size, trg_vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        
    def make_src_mask(self, src):
        # padding mask
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
        return src_mask
    
    def forward(self, src, trg):
        src_seq_length = src.shape[1]
        trg_seq_length = trg.shape[1]
        
        # 位置编码
        src_positions = torch.arange(0, src_seq_length).unsqueeze(0).expand(src.size(0), src_seq_length).to(src.device)
        trg_positions = torch.arange(0, trg_seq_length).unsqueeze(0).expand(trg.size(0), trg_seq_length).to(trg.device)
        
        src_embed = self.dropout(self.src_embed(src) + self.pos_encoder(src_positions))
        trg_embed = self.dropout(self.trg_embed(trg) + self.pos_encoder(trg_positions))
        
        src_mask = self.make_src_mask(src)
        
        # 通过Transformer块
        x = src_embed
        for block in self.transformer_blocks:
            x = block(x, src_mask)
        
        # 解码器自注意力mask
        trg_mask = torch.tril(torch.ones(trg_seq_length, trg_seq_length)).unsqueeze(0).unsqueeze(1).to(trg.device)
        
        # 通过Transformer块（解码器部分简化为同一模块）
        x = trg_embed
        for block in self.transformer_blocks:
            x = block(x, trg_mask)
        
        out = self.fc_out(x)
        return out

# 模型参数示例
src_vocab_size = 10000
trg_vocab_size = 10000
src_pad_idx = 0
trg_pad_idx = 0

model = Transformer(src_vocab_size, trg_vocab_size, src_pad_idx, trg_pad_idx)
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")


## 3. 代码示例


In [ ]:
"""
L14-attention-visualization.py
注意力可视化：展示Transformer学到的注意力模式
"""

import torch
import matplotlib.pyplot as plt
import seaborn as sns

def visualize_attention(attention_weights, tokens, layer_idx=0, head_idx=0):
    """
    可视化单个注意力头的权重
    attention_weights: (batch, heads, seq_len, seq_len)
    """
    # 取指定层和头的注意力权重
    attn = attention_weights[0, head_idx].detach().numpy()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(attn, xticklabels=tokens, yticklabels=tokens, cmap='viridis')
    plt.title(f'Layer {layer_idx}, Head {head_idx} Attention')
    plt.xlabel('Key Tokens')
    plt.ylabel('Query Tokens')
    plt.tight_layout()
    plt.savefig(f"l14_attention_layer{layer_idx}_head{head_idx}.png", dpi=150)
    plt.show()

# 演示不同类型的注意力模式
print("Transformer注意力模式类型:")
print("=" * 50)
print("1. 对角线注意力：每个位置关注自身附近")
print("2. 全部注意力：所有位置关注[CLS]或特定标记")
print("3. 垂直注意力：关注句子中的特定词（如动词、名词）")
print("4. 稀疏注意力：形成多个局部注意力簇")
print("\n这些模式反映了不同层的功能：")
print("  - 浅层：更多关注局部语法结构")
print("  - 深层：更多关注语义关系和远程依赖")

# 模拟不同层的注意力模式
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

layers = ['Layer 1 (浅层)', 'Layer 4 (中层)', 'Layer 8 (深层)', 'Layer 12 (顶层)']
attention_types = ['local', 'syntactic', 'semantic', 'global']

for idx, (layer, attn_type) in enumerate(zip(layers, attention_types)):
    ax = axes[idx // 2, idx % 2]
    
    # 模拟不同类型的注意力矩阵
    seq_len = 20
    if attn_type == 'local':
        # 局部注意力：关注附近位置
        attn = np.zeros((seq_len, seq_len))
        for i in range(seq_len):
            for j in range(max(0, i-3), min(seq_len, i+4)):
                attn[i, j] = 1 / (abs(i-j) + 1)
    
    elif attn_type == 'syntactic':
        # 句法注意力：关注相邻词
        attn = np.eye(seq_len)
        for i in range(seq_len-1):
            attn[i, i+1] = 0.5
            attn[i+1, i] = 0.5
    
    elif attn_type == 'semantic':
        # 语义注意力：某些词（索引5和15）被广泛关注
        attn = np.ones((seq_len, seq_len)) * 0.1
        for i in range(seq_len):
            attn[i, 5] = 0.8
            attn[i, 15] = 0.8
    
    else:  # global
        # 全局注意力：均匀分布
        attn = np.ones((seq_len, seq_len)) / seq_len
    
    # 归一化
    attn = attn / attn.sum(axis=1, keepdims=True)
    
    sns.heatmap(attn, ax=ax, cmap='YlOrRd')
    ax.set_title(f'{layer}\n({attn_type} pattern)')
    ax.set_xlabel('Key')
    ax.set_ylabel('Query')

plt.suptitle('不同层的注意力模式', fontsize=14)
plt.tight_layout()
plt.savefig("l14_attention_patterns.png", dpi=150, bbox_inches='tight')
plt.show()


## 4. 练习题

1. **注意力计算**: 给定Q、K、V矩阵，手动计算自注意力的输出，包括scaled dot-product、softmax、加权求和。
2. **多头注意力分析**: 解释为什么多头注意力比单头更有效——不同头是否学习到不同的表示？
3. **位置编码设计**: 设计一种可学习的位置编码方案，与正弦/余弦位置编码比较优缺点。
4. **Transformer实现**: 使用PyTorch实现一个简化的Transformer Encoder（不含Decoder），在文本分类任务上验证。
5. **注意力可视化**: 对预训练的BERT模型，可视化其在不同NLP任务上的注意力模式，分析哪些头学习到了语言知识。

## 5. 延伸阅读

- **论文**: Vaswani et al., 2017, "Attention Is All You Need" — Transformer原始论文
- **教程**: The Illustrated Transformer — https://jalammar.github.io/illustrated-transformer/
- **教程**: The Annotated Transformer — https://nlp.seas.harvard.edu/2018/04/03/attention.html
- **代码**: HuggingFace Transformers库 — https://github.com/huggingface/transformers
- **视频**: Stanford CS224N Lecture 11 — Attention and Transformers

---
